In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.odr import ODR, Model, RealData
from math import floor, log10

excel_file = "newtons_second_law_data_table.xlsx"
sheet_name_1 = "constant_mass"
sheet_name_2 = "constant_force"

def constant_mass_analysis(excel_file, sheet_name):
    df = pd.read_excel(excel_file, sheet_name=sheet_name)
    df_constants = pd.read_excel(excel_file, sheet_name="constants")

    df["force"] = df["Hanging Mass (kg)"] * 9.81

    y_error = 9.81*0.001
    df["force_error"] = y_error

    x = df["Acceleration (m/s2)"]
    y = df["force"]

    xerr = df["Acceleration Error"]
    yerr = df["force_error"]

    
    # Linear model for ODR
    def linear(beta, x):
        return beta[0] * x + beta[1]

    # Create model and data objects
    model = Model(linear)
    data = RealData(x, y, sx=xerr, sy=yerr)

    # Initial guesses for slope and intercept
    odr = ODR(data, model, beta0=[1.0, 0.0])

    # Run fit
    output = odr.run()

    # Best-fit parameters
    m, b = output.beta

    # Parameter uncertainties
    m_err, b_err = output.sd_beta

    # Generate fit line
    x_fit = np.linspace(min(x)-0.02, max(x)+0.02, 100)
    y_fit = linear([m, b], x_fit)

    # Plot
    plt.figure(figsize=(8,6))

    plt.errorbar(
    x,
    y,
    xerr=xerr,
    yerr=yerr,
    fmt='o',
    capsize=4,
    color='blue',
    label='Experimental Data'
    )

    plt.plot(
    x_fit,
    y_fit,
    'r-',
    linewidth=2,
    label=(
    f'm = {m:.3f} ± {m_err:.3f}\n'
    f'b = {b:.3f} ± {b_err:.3f}'
    )
    )

    plt.xlabel(r'Acceleration (m/s$^2$)', fontsize=14)
    plt.ylabel('Force (N)', fontsize=14)
    plt.title('Force vs Acceleration')
    plt.grid(True)
    plt.legend()
    plt.tight_layout()

    plt.savefig(f"{sheet_name}.png", dpi=300, bbox_inches="tight")

    plt.show()

    print(f"Analysis for {sheet_name}:")

    mass = m
    mass_err = m_err

    # Determine decimal place corresponding to 1 significant digit in the error
    decimals = -floor(log10(abs(mass_err)))

    # Round error to 1 significant digit
    mass_err_rounded = round(mass_err, decimals)

    # Round value to same decimal place
    mass_rounded = round(mass, decimals)

    print(f"total_mass = {mass_rounded:.{decimals}f} ± {mass_err_rounded:.{decimals}f} kg")

    measured_mass = float(df_constants["constant mass"][0])

    percent_difference = abs(mass - measured_mass) / measured_mass * 100

    print(f"Percentage difference = {percent_difference:.2f}%")

    

def constant_force_analysis(excel_file, sheet_name):
    df = pd.read_excel(excel_file, sheet_name=sheet_name)
    df_constants = pd.read_excel(excel_file, sheet_name="constants")

    df["1/m"] = 1/df["Total System Mass (Kg)"]

    df["mass_error"] = 0.00001/(df["Total System Mass (Kg)"]**2)

    x = df["1/m"]
    y = df["Acceleration (m/s2)"]

    xerr = df["mass_error"]
    yerr = df["Acceleration Error"]

    # Linear model for ODR
    def linear(beta, x):
        return beta[0] * x + beta[1]

    # Create model and data objects
    model = Model(linear)
    data = RealData(x, y, sx=xerr, sy=yerr)

    # Initial guesses for slope and intercept
    odr = ODR(data, model, beta0=[1.0, 0.0])

    # Run fit
    output = odr.run()

    # Best-fit parameters
    m, b = output.beta

    # Parameter uncertainties
    m_err, b_err = output.sd_beta

    # Generate fit line
    x_fit = np.linspace(min(x)-0.02, max(x)+0.02, 100)
    y_fit = linear([m, b], x_fit)

    # Plot
    plt.figure(figsize=(8,6))

    plt.errorbar(
    x,
    y,
    xerr=xerr,
    yerr=yerr,
    fmt='o',
    capsize=4,
    color='blue',
    label='Experimental Data'
    )

    plt.plot(
    x_fit,
    y_fit,
    'r-',
    linewidth=2,
    label=(
    f'm = {m:.3f} ± {m_err:.3f}\n'
    f'b = {b:.3f} ± {b_err:.3f}'
    )
    )

    plt.xlabel(r'Mass$^{-1}$ (kg$^{-1}$)', fontsize=14)
    plt.ylabel(r'Acceleration (m/s$^2$)', fontsize=14)
    plt.title(r'Acceleration vs Mass$^{-1}$')
    plt.grid(True)
    plt.legend()
    plt.tight_layout()

    plt.savefig(f"{sheet_name}.png", dpi=300, bbox_inches="tight")

    plt.show()

    print(f"Analysis for {sheet_name}:")

    mass = m/9.81
    mass_err = m_err/9.81

    # Determine decimal place corresponding to 1 significant digit in the error
    decimals = -floor(log10(abs(mass_err)))

    # Round error to 1 significant digit
    mass_err_rounded = round(mass_err, decimals)

    # Round value to same decimal place
    mass_rounded = round(mass, decimals)

    print(f"hanging_mass = {mass_rounded:.{decimals}f} ± {mass_err_rounded:.{decimals}f} kg")

    measured_mass = float(df_constants["constant hanging mass"][0])

    percent_difference = abs(mass - measured_mass) / measured_mass * 100

    print(f"Percentage difference = {percent_difference:.2f}%")

constant_mass_analysis(excel_file, sheet_name_1)
constant_force_analysis(excel_file, sheet_name_2)